# 04. 재구매 — 답이 안 나올 수도 있는 질문> **낮은 리뷰 점수가 재구매 감소로 이어지는가?**## 시작 전에 각오할 것이 데이터는 반복 구매 비중이 매우 낮습니다.그래서 이 질문은 **"표본이 부족해 판단할 수 없다"가 결론일 가능성이 높습니다.**그때 숫자를 억지로 만들지 마세요.표본 수와 함께 "이 데이터로는 답할 수 없다"를 근거 있게 적는 것이 정답이고,**그렇게 쓸 줄 아는지가 이 프로젝트에서 가장 크게 평가받는 지점입니다.**대부분의 포트폴리오는 잘 나온 결과만 싣습니다. 한계를 근거와 함께 적은 문서는 드물고,현업에서는 그쪽이 더 신뢰를 받습니다.

In [ ]:
import duckdb, pandas as pdimport matplotlib, matplotlib.pyplot as pltpd.set_option("display.max_columns", 50)# 한글 깨짐 방지for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR", "Noto Sans KR"]:    if _f in {f.name for f in matplotlib.font_manager.fontManager.ttflist}:        plt.rcParams["font.family"] = _f        breakplt.rcParams["axes.unicode_minus"] = False# DuckDB는 CSV를 그대로 읽습니다. 적재 과정이 따로 없습니다.con = duckdb.connect()con.execute(open("../sql/base.sql", encoding="utf-8").read().format(data="../data/raw"))def q(sql):    """SQL을 실행해 pandas DataFrame으로 돌려줍니다."""    return con.sql(sql).df()q("SELECT count(*) AS 주문수 FROM order_fact")

## 1. 먼저 표본이 있는지부터 확인분석을 설계하기 전에 **애초에 답할 수 있는 질문인지** 를 봅니다.여기서 나온 숫자가 이 노트북의 결론을 사실상 결정합니다.

In [ ]:
q("""WITH per_person AS (    SELECT customer_unique_id, count(*) AS n    FROM order_fact    WHERE customer_unique_id IS NOT NULL    GROUP BY 1)SELECT count(*)                                                    AS 고객수,       count(*) FILTER (WHERE n >= 2)                              AS 재구매고객수,       round(100.0 * count(*) FILTER (WHERE n >= 2) / count(*), 2) AS 재구매율_퍼센트,       max(n)                                                      AS 최다주문수FROM per_person""")

**판단 지점**재구매 고객 수가 몇 명인가요? 이걸 리뷰 점수 5단계로 나누면 한 칸에 몇 명이 남나요?한 칸에 수십 명 수준이라면, 점수별 재구매율 차이는 우연으로도 충분히 나올 수 있는 크기입니다.

## 2. 【직접 작성】 LEAD로 다음 주문 붙이기`LEAD()` 는 **같은 그룹 안에서 다음 행의 값** 을 현재 행에 붙여 줍니다.고객별로 주문을 시간순 정렬한 뒤 "다음 주문이 있는가"를 판정하는 데 씁니다.```sqlLEAD(purchased_at) OVER (PARTITION BY customer_unique_id ORDER BY purchased_at)```- `PARTITION BY` = 고객 단위로 끊어서- `ORDER BY` = 그 안에서 시간순으로- 다음 주문이 없으면 `NULL` 이 들어옵니다 → 그게 "재구매 안 함"### 만들 것1. 고객별 주문에 `다음주문시점` 을 붙인다2. 각 고객의 **첫 주문** 만 남긴다 (`ROW_NUMBER() ... = 1`)3. 첫 주문의 리뷰 점수별로 재구매 비율을 집계한다4. **각 점수 구간의 표본 수를 반드시 함께 출력한다** ← 이게 결론의 근거가 됩니다

In [ ]:
# 여기에 작성하세요# q("""# WITH seq AS (#     SELECT customer_unique_id, order_id, purchased_at, review_score,#            LEAD(purchased_at) OVER (PARTITION BY customer_unique_id ORDER BY purchased_at) AS 다음주문,#            ROW_NUMBER()       OVER (PARTITION BY customer_unique_id ORDER BY purchased_at) AS rn#     FROM order_fact#     WHERE customer_unique_id IS NOT NULL# )# SELECT review_score AS 첫주문_점수,#        count(*) AS 고객수,#        count(*) FILTER (WHERE 다음주문 IS NOT NULL) AS 재구매고객수,#        round(100.0 * count(*) FILTER (WHERE 다음주문 IS NOT NULL) / count(*), 2) AS 재구매율# FROM seq# WHERE rn = 1 AND review_score IS NOT NULL# GROUP BY 1 ORDER BY 1# """)

## 3. 결론 쓰기 — 두 갈래### (A) 표본이 충분했다면> 첫 주문에 **[   ]** 점을 준 고객의 재구매율은 **[  ]** %(n=**[   ]**),> **[   ]** 점을 준 고객은 **[  ]** %(n=**[   ]**)로 **[   ]** %p 차이가 있다.### (B) 표본이 부족했다면 — 이렇게 쓰세요> 재구매 고객은 전체 **[   ]** 명 중 **[   ]** 명(**[  ]**%)에 그쳤다.> 이를 리뷰 점수 5단계로 나누면 구간당 **[   ]** 명 수준이어서,> 점수별 재구매율 차이를 유의미하게 말할 수 없다.> **따라서 이 데이터로는 "낮은 점수가 이탈로 이어지는가"에 답할 수 없다고 판단하고 결론을 보류한다.**> 이 질문에 답하려면 동일 고객의 장기 구매 이력이 필요하다.(B)를 쓰는 걸 실패로 생각하지 마세요.**답이 안 나온다는 것을 근거와 함께 밝히는 것도 분석 결과입니다.**`docs/FINDINGS.md` 의 "확인하지 못한 것" 섹션이 그걸 위해 만들어져 있습니다.